# 01 — CPU, GPU, and Accelerator Mental Models

**Master syllabus coverage:** V2 1.1

## Why this matters

SLM workloads mix control-heavy host work with large parallel tensor operations. Knowing which device is suited to which work prevents the vague and often false rule that a GPU is always faster.

## Learning objectives

- Contrast CPU latency-oriented execution with GPU throughput-oriented execution.
- Separate hardware differences from interpreter and kernel implementation differences.
- Measure asynchronous device work with explicit synchronization.
- Distinguish latency, throughput, transfer cost, and workload size.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Different hardware optimizes different work

A CPU has relatively few sophisticated cores optimized for low-latency control flow,
branch prediction, and varied tasks. A GPU has many simpler execution lanes optimized
for high-throughput data-parallel arithmetic. An accelerator is useful only when the
workload exposes enough parallel work to repay launch, scheduling, and transfer costs.

Integrated GPUs such as Apple Silicon share a physical memory pool with the CPU, while
discrete GPUs have separate device memory. Shared physical memory does not mean every
framework operation or synchronization is free.


In [ ]:
import platform
import time
import numpy as np
import torch

def available_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = available_device()
print("machine:", platform.machine())
print("processor:", platform.processor() or "not reported")
print("PyTorch threads:", torch.get_num_threads())
print("selected accelerator:", device)


## 2. Python loop versus compiled vector kernel

Comparing a Python loop with NumPy does not isolate CPU versus GPU—the NumPy operation
is compiled, vectorized, and often multithreaded. It demonstrates why execution strategy
matters even on identical hardware. Always name every difference in a benchmark.


In [ ]:
rng = np.random.default_rng(42)
left = rng.normal(size=200_000).astype(np.float32)
right = rng.normal(size=200_000).astype(np.float32)

start = time.perf_counter()
loop_result = np.empty_like(left)
for index in range(left.size):
    loop_result[index] = left[index] + right[index]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_result = left + right
vector_time = time.perf_counter() - start
np.testing.assert_array_equal(loop_result, vector_result)
print(f"Python loop={loop_time*1e3:.2f} ms, NumPy kernel={vector_time*1e3:.3f} ms")


## 3. Launch and synchronization change measured latency

Accelerator operations are commonly asynchronous: the host queues work and continues.
Synchronize immediately before starting and after ending a timed region. Time transfers
separately when they are part of the user-visible operation.


In [ ]:
def synchronize(target: torch.device) -> None:
    if target.type == "cuda":
        torch.cuda.synchronize(target)
    elif target.type == "mps":
        torch.mps.synchronize()

cpu_tensor = torch.from_numpy(left)
start = time.perf_counter()
device_tensor = cpu_tensor.to(device)
synchronize(device)
transfer_time = time.perf_counter() - start

for _ in range(3):
    _ = device_tensor + 1
synchronize(device)
start = time.perf_counter()
result = device_tensor + 1
synchronize(device)
kernel_time = time.perf_counter() - start
print(f"CPU→{device} transfer={transfer_time*1e3:.3f} ms, one kernel={kernel_time*1e3:.3f} ms")
assert result.shape == device_tensor.shape


## 4. Latency and throughput are different objectives

Latency measures time for one request; throughput measures work per unit time. Batching
often improves throughput by exposing more parallelism while increasing the time one
request waits. Humor Machine will care about model-load time, time to first token, decode
tokens per second, total response latency, and interface responsiveness.


In [ ]:
def timed_matmul(size: int, repeats: int = 5) -> tuple[float, float]:
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)
    for _ in range(2):
        _ = a @ b
    synchronize(device)
    start = time.perf_counter()
    for _ in range(repeats):
        output = a @ b
    synchronize(device)
    seconds = (time.perf_counter() - start) / repeats
    operations = 2 * size**3
    assert torch.isfinite(output).all()
    return seconds, operations / seconds

for size in (32, 128, 512):
    seconds, operations_per_second = timed_matmul(size)
    print(f"n={size:3}: latency={seconds*1e3:8.3f} ms, rough throughput={operations_per_second/1e9:7.2f} GFLOP/s")


## Failure modes to recognize

- **GPU-always-faster assumption:** tiny or control-heavy workloads lose to overhead.
- **Unsynchronized timing:** the measurement captures queueing rather than completion.
- **Transfer omitted:** an accelerator benchmark excludes a cost the application must pay.
- **Confounded comparison:** Python versus compiled code is described as only CPU versus GPU.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Benchmark vector addition at four sizes on CPU and your available accelerator and identify the crossover, if any.
2. Measure one matmul versus a batch of matmuls and compare latency with throughput.
3. Draw the CPU/accelerator ownership path for loading data, training, checkpointing, and generation.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can predict which workload properties favor a CPU or accelerator and design a timing that includes synchronization and transfers.

**Next:** 02 — Memory hierarchy and data movement.
